#### 1. Librerías.

In [ ]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [ ]:
#a. Modo de ejecución (se define en ./constantes/modo.txt, un solo lugar para los 4 notebooks).
# "validacion" = entreno solo con train, puedo medir nDCG.
# "entrega"    = entreno con train+test, uso todo el historial para predecir.
with open("./constantes/modo.txt") as f:
    MODO = f.read().strip()

assert MODO in ("validacion", "entrega"), f"MODO inválido: {MODO!r}"
print(f"MODO: {MODO}")

In [ ]:
#b. Otras constantes.
%run "./constantes/constantes.ipynb"

In [ ]:
#c. Verificación del modo (que los paths coincidan con lo que creo que estoy corriendo).
print(f"MODO: {MODO} | sufijo: {sufijo!r}")
print(f"train_fe: {path_train_fe}")
print(f"modelo:   {path_modelo}")

#### 3. Funciones.

In [ ]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [ ]:
#a. Train.
# Fuerzo los ids a str: si el CSV los infiere como int, los merges contra
# df_libros / df_lectores devuelven NaN en silencio.
df_train = pd.read_csv(path_train_fe, dtype={"id_lector": str, "id_libro": str})

In [ ]:
#b. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir, dtype={"id_lector": str})

In [ ]:
#c. Libros y Lectores (lo tomo para armar la predicción).
df_libros = pd.read_csv(path_libros_fe, dtype={"id_libro": str})
df_lectores = pd.read_csv(path_lectores_fe, dtype={"id_lector": str})

In [ ]:
#d. Leo el modelo (LGBMRanker entrenado en el notebook 3).
modelo = joblib.load(path_modelo_lgbmranker)
print(f"Modelo: {type(modelo).__name__} | features que espera: {modelo.n_features_in_}")

#### 5. Preparación previa.

In [ ]:
#a. Armo la lista de features.
# TIENE que ser idéntica a la del notebook 3: mismas columnas, mismo orden. Si no coincide,
# el modelo predice sobre otra cosa. El assert de abajo lo verifica contra el modelo mismo.
features_base = [
    "anio_edicion", 
    "nacimiento",
    #"edad_al_interactuar",            # En train varía con fecha.dt.year, al predecir es
    #"dias_transcurridos_interaccion", # anio_actual - nacimiento (constante por lector).
    #"anios_transcurridos_edicion",    # Idem: al predecir queda determinada por anio_edicion.
    #"antiguedad_libro_hoy",
    'frecuencia_lector', 
    'frecuencia_libro', 
    'n_lectores_distintos_autor',
    #'n_interacciones_lector_autor',   # reemplazada por prop_lector_autor (dan el mismo nCDG).
    #'n_interacciones_lector_genero',  # reemplazada por prop_lector_genero (dan el mismo nCDG).
    #'n_autores_distintos_lector',     # reemplazada por prop_autores_distintos (dan el mismo nCDG).
    #'n_generos_distintos_lector',     # reemplazada por prop_generos_distintos (dan el mismo nCDG).
    #'rating_prom_id_lector', 
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado', 
    # Las tres de calidad de ítem degradaban el ranking con RF sobre RMSE
    # (0.0697 -> 0.0409). Con LGBM Ranker pasa algo parecido:
    # La mejor feature para predecir rating es la peor para rankear, 
    # y cambiar el objetivo mitiga pero no elimina el conflicto.
    #'rating_prom_id_libro',
    #'rating_prom_autor',
    #'rating_prom_genero',
    'prop_lector_autor',
    'prop_lector_genero',
    'prop_autores_distintos',
    'prop_generos_distintos',
    'log_pop_media_lector',   # tanda B
    'dif_log_pop',            # tanda B
]
features_dummies = [c for c in df_train.columns if c.startswith((
    "genero_persona_", 
    #"genero_libro_agrupado_", 
    #"editorial_agrupada_", 
    #"pais_agrupado_"
))]
features = features_base + features_dummies

print(f"{len(features)} features:", features)

In [ ]:
#b. Verifico que las features coincidan con las que vio el modelo al entrenar.
# Este assert es lo que evita el error mas caro posible en este notebook: entrenar con un
# set de columnas y predecir con otro. Falla ruidoso en vez de generar un entregable malo.
if hasattr(modelo, "feature_name_"):
    esperadas = list(modelo.feature_name_)
    assert features == esperadas, (
        f"Las features no coinciden con las del modelo.\n"
        f"Sobran: {set(features) - set(esperadas)}\n"
        f"Faltan: {set(esperadas) - set(features)}\n"
        f"Mismo set pero distinto orden: {sorted(features) == sorted(esperadas)}"
    )
    print("Features alineadas con el modelo: ok")
else:
    assert len(features) == modelo.n_features_in_, (
        f"El modelo espera {modelo.n_features_in_} features y le paso {len(features)}"
    )
    print("Cantidad de features alineada con el modelo: ok")

#### 6. Predicción.

In [ ]:
#a. Me aseguro de que todos los lectores a predecir tengan fila en df_lectores.
# Kaggle pide 832 lectores; si alguno no está en la tabla de lectores, el merge de
# caract_lector no matchea y sus features quedan en NaN. No los puedo descartar como a
# los libros sin metadata: hay que entregar recomendaciones para todos.
faltantes = set(df_a_predecir["id_lector"]) - set(df_lectores["id_lector"])
if faltantes:
    print(f"Agrego {len(faltantes)} lectores sin fila en df_lectores: {sorted(faltantes)}")
    df_lectores = pd.concat(
        [df_lectores, pd.DataFrame({"id_lector": sorted(faltantes)})],
        ignore_index=True
    )

In [ ]:
#a. Tablas de referencia para el feature engineering de los candidatos.
# Van primero porque el filtro del universo de libros las necesita.
# feature_engineering_test las toma como globales, así que los nombres del desempaquetado
# tienen que quedar exactamente así.
(media_global, caract_lector, caract_libros,
 afinidad_lector_autor_test, afinidad_lector_genero_test) = armar_tablas_referencia(
    df_train, df_libros, df_lectores
)

In [ ]:
#b. Universo de libros candidatos.
#i. Todos los libros con al menos una interacción.
conn = sqlite3.connect(path_db)
todos_los_libros = pd.read_sql("SELECT id_libro FROM interacciones", conn)["id_libro"].unique()
conn.close()

#ii. Filtro los libros sin la metadata que necesitan las features. Dos casos:
# - ~70 libros que están en interacciones pero no tienen fila en df_libros.
# - ~4 libros que sí están pero con autor en NaN: el merge de afinidad se hace sobre
#   ["id_lector", "autor"], y NaN nunca matchea contra NaN.
# Sin este filtro llegan al modelo con NaN y se rutean a un lado arbitrario del árbol,
# sin tirar error. Es el mismo arreglo que en el notebook 3.
n_antes = len(todos_los_libros)
libros_con_metadata = set(
    caract_libros.dropna(subset=["autor", "genero_libro_agrupado", "anio_edicion"])["id_libro"]
)
todos_los_libros = np.array([b for b in todos_los_libros if b in libros_con_metadata])

print(f"Universo de candidatos: {len(todos_los_libros):,} "
      f"(descarto {n_antes - len(todos_los_libros)} sin metadata)")

In [ ]:
#c. Historial por lector (lo usa retrieval para excluir lo ya leído).
leidos_por_lector = (
    df_train[["id_lector", "id_libro"]]
    .groupby("id_lector")["id_libro"]
    .apply(set)
    .to_dict()
)

#d. Lectores a recomendarle.
id_lectores_predecir = df_a_predecir["id_lector"].unique()

In [ ]:
#e. Verificación: los lectores a predecir, ¿tienen historial?
freq = df_train.groupby("id_lector").size()
cobertura = df_a_predecir["id_lector"].map(freq)
print(f"Modo: {MODO} | Filas de la base: {len(df_train):,}")
print("Lectores pedidos:", len(id_lectores_predecir))
print("Sin historial:", cobertura.isna().sum())
print("Mediana de frecuencia:", cobertura.median())

In [ ]:
#f. Calculamos ranking final para producción.
#i. Lista donde almacenaremos las recomendaciones.
recomendaciones = []
sin_candidatos = []
total_lectores = len(id_lectores_predecir)

print("Comienza la predicción general.")

#ii. Recorro cada id_lector.
warnings.filterwarnings("ignore", message=".*eval_at.*")

for i, id_lector in enumerate(id_lectores_predecir, start=1):
    if i % 50 == 0 or i == 1:
        print("Lector {}/{}".format(i, total_lectores))

    #1. Retrieval.
    libros_candidatos_a_recomendar = retrieval(id_lector)

    # Si no tiene candidatos queda fuera del entregable: lo registro en vez de saltearlo
    # en silencio, porque Kaggle promedia sobre TODOS los lectores pedidos y un lector
    # faltante suma un 0 al promedio.
    if len(libros_candidatos_a_recomendar) == 0:
        sin_candidatos.append(id_lector)
        continue

    #2. Feature engineering sobre todos los candidatos.
    df_features_candidatos = feature_engineering_test(id_lector, libros_candidatos_a_recomendar)

    #3. Ningún NaN puede llegar al modelo: sería un merge fallido, no un dato faltante.
    if df_features_candidatos[features].isna().any().any():
        cols = df_features_candidatos[features].isna().sum()
        raise ValueError(f"NaN en features para {id_lector}: {cols[cols > 0].to_dict()}")

    #4. Predigo el score de ranking de cada candidato.
    # OJO: con el ranker esto ya no es un rating predicho (escala 1-10) sino un score de
    # relevancia sin escala interpretable — puede ser negativo. No importa: solo lo uso
    # para ordenar.
    X = df_features_candidatos[features]
    df_features_candidatos["score_ranking"] = modelo.predict(X)

    #5. Ordeno de mayor a menor y me quedo con los 20 mejores.
    top_20 = df_features_candidatos.sort_values("score_ranking", ascending=False).head(20)

    #6. Guardo las recomendaciones (el ORDEN de las filas es la métrica: nDCG mide posición).
    recomendaciones.append(top_20[["id_lector", "id_libro"]])

#iii. Uno todas las recomendaciones.
df_recomendaciones = pd.concat(recomendaciones, ignore_index=True)

print("\nPredicción general finalizada.")
print("Cantidad de recomendaciones:", len(df_recomendaciones))
if sin_candidatos:
    print(f"ATENCIÓN: {len(sin_candidatos)} lectores sin candidatos, quedan fuera:", sin_candidatos[:10])

#### 7. Exportación de las recomendaciones.

In [ ]:
#a. Comprobaciones antes de exportar.
assert df_recomendaciones["id_lector"].nunique() == len(id_lectores_predecir), (
    f"Faltan lectores: entrego {df_recomendaciones['id_lector'].nunique()} "
    f"de {len(id_lectores_predecir)}"
)
assert (df_recomendaciones.groupby("id_lector").size() == 20).all(), (
    "Hay lectores con distinta cantidad de 20 recomendaciones"
)
assert df_recomendaciones.duplicated(["id_lector", "id_libro"]).sum() == 0, (
    "Hay pares (lector, libro) duplicados"
)
print("Lectores entregados:", df_recomendaciones["id_lector"].nunique())
print("Filas por lector:", df_recomendaciones.groupby("id_lector").size().unique())
print("Comprobaciones: ok")

In [ ]:
#b. Defino el número de versión (a mano).
version = "16"

In [ ]:
#c. Exporto la versión de esta corrida.
# index=False y sin reordenar: el orden de las filas dentro de cada lector ES el ranking.
ruta = f"./outputs/entregable_{version}.csv"
df_recomendaciones.to_csv(ruta, index=False)
print(f"Exportado: {ruta} ({len(df_recomendaciones):,} filas)")

In [ ]:
#d. Compruebo el archivo escrito (releo, por las dudas).
chequeo = pd.read_csv(ruta, dtype={"id_lector": str, "id_libro": str})
print("Lectores pedidos:   ", df_a_predecir["id_lector"].nunique())
print("Lectores entregados:", chequeo["id_lector"].nunique())
print(chequeo.groupby("id_lector").size().value_counts())
print("\nPrimeras filas:")
print(chequeo.head(5))